# Spike: MLflow storage on Google Drive (ADR-001)

Question: can an MLflow store (SQLite database plus file artifacts) live on Google Drive reliably?

Run the cells **in order**. Cell 4 needs a runtime restart in the middle. Paste every `RESULT:` line and any error text back to the assistant.

Pass criteria are in `docs/decisions/ADR-001-mlflow-storage.md`.

In [ ]:
# --- AI Experiment Lab: standard Colab setup cell ---------------------------------
import os, subprocess, sys

REPO_URL = "https://github.com/<your-username>/ai-experiment-lab.git"   # <-- edit
REPO_DIR = "/content/ai-experiment-lab"
DRIVE_ROOT = "/content/drive/MyDrive/ai-experiment-lab"
EXTRAS = "ml,tracking"          # pip extras from pyproject.toml

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlflow", exist_ok=True)

# For a private repo, store a token in Colab Secrets as GITHUB_TOKEN (never paste it here).
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# An editable install adds a .pth file, which a running kernel does not re-read.
# Put src/ on sys.path so the import works in this session without a restart.
src_dir = f"{REPO_DIR}/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Tracking location: persistent Drive storage (see docs/decisions/ADR-001-mlflow-storage.md).
os.environ["MLFLOW_TRACKING_URI"] = f"file:{DRIVE_ROOT}/mlflow"
os.environ["AEL_DATA_DIR"] = f"{DRIVE_ROOT}/data"

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code version (commit):", commit)
print("Data dir:", os.environ["AEL_DATA_DIR"])

import ai_experiment_lab
print("ai_experiment_lab", ai_experiment_lab.__version__, "imported OK")

In [ ]:
# Cell 2: baseline on Colab's LOCAL disk (fast, not persistent). For timing comparison only.
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
SPIKE = f"{REPO_DIR}/scripts/mlflow_storage_spike.py"
!python {SPIKE} write --root /content/_spike_local --n 30

In [ ]:
# Cell 3: the real test. Same workload on Google Drive.
!python {SPIKE} write --root {DRIVE_ROOT}/spike --n 30

In [ ]:
# Cell 3b: two processes writing at the same time to the Drive store.
!python {SPIKE} concurrent --root {DRIVE_ROOT}/spike --n 15

## Cell 4: persistence across a runtime restart

1. Menu: **Runtime > Disconnect and delete runtime**.
2. Reconnect, then run the **setup cell** again (it re-mounts Drive).
3. Run the verify cell below.

This proves the data survives the loss of the Colab machine, which is the whole point.

In [ ]:
# Cell 5 (after restart + setup cell): is everything still there?
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
SPIKE = f"{REPO_DIR}/scripts/mlflow_storage_spike.py"
!python {SPIKE} verify --root {DRIVE_ROOT}/spike

## Cleanup (optional)

Once results are recorded, delete the `spike` folder in `MyDrive/ai-experiment-lab/` from Drive.